# Runbook RAG Chat

RAG over internal runbook documents: **no LangChain**. We load Markdown from `knowledge_base/`, chunk, embed with **OpenAI**, store in **ChromaDB**, and answer questions with **OpenAI chat** using retrieved context.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
import gradio as gr

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENROUTER_API_KEY")
base_url = os.getenv("OPENROUTER_BASE_URL")
if not OPENAI_API_KEY:
    print("Set OPENROUTER_API_KEY in .env")
client = OpenAI(base_url=base_url, api_key=OPENAI_API_KEY)

CHUNK_SIZE = 600
CHUNK_OVERLAP = 80
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
COLLECTION_NAME = "runbooks"
CHROMA_PATH = "chroma_runbooks"
KNOWLEDGE_BASE = Path("knowledge_base")

## 1. Load and chunk documents

In [ ]:
def load_docs(base_path: Path):
    """Load all .md files under base_path. Returns list of (text, source)."""
    docs = []
    for f in base_path.rglob("*.md"):
        text = f.read_text(encoding="utf-8")
        docs.append((text, str(f)))
    return docs

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start = end - overlap
    return chunks

documents = load_docs(KNOWLEDGE_BASE)
all_chunks = []
for text, source in documents:
    for i, c in enumerate(chunk_text(text)):
        all_chunks.append({"content": c, "source": source})
print(f"Loaded {len(documents)} docs, {len(all_chunks)} chunks")

## 2. Embed and store in Chroma

In [ ]:
def get_embeddings(texts: list[str]):
    """OpenAI embeddings for a list of texts."""
    out = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [e.embedding for e in out.data]

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(COLLECTION_NAME)
collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"description": "Runbook chunks"})

# Batch embed and add (OpenAI allows many inputs per request)
texts = [c["content"] for c in all_chunks]
batch_size = 100
for i in range(0, len(texts), batch_size):
    batch = texts[i : i + batch_size]
    embeddings = get_embeddings(batch)
    ids = [f"chunk_{i + j}" for j in range(len(batch))]
    metadatas = [{"source": all_chunks[i + j]["source"]} for j in range(len(batch))]
    collection.add(ids=ids, embeddings=embeddings, documents=batch, metadatas=metadatas)
print(f"Chroma collection '{COLLECTION_NAME}' has {collection.count()} chunks.")

## 3. Retrieve and answer (RAG)

In [ ]:
SYSTEM_PROMPT = """You are a runbook assistant. Answer only from the provided context (internal runbooks). Be concise. If the context does not contain enough information, say so. Do not make up steps or commands."""

def retrieve(query: str, n: int = 4):
    """Return top-n chunks for the query."""
    q_emb = get_embeddings([query])[0]
    result = collection.query(query_embeddings=[q_emb], n_results=min(n, collection.count()), include=["documents", "metadatas"])
    return result["documents"][0], result["metadatas"][0]

def rag_chat(message: str, history: list):
    """Answer using retrieved runbook context."""
    docs, metas = retrieve(message)
    context = "\n\n---\n\n".join(docs) if docs else "(No relevant runbook sections found.)"
    user_content = f"Context from runbooks:\n\n{context}\n\n---\n\nUser question: {message}"
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_content}]
    resp = client.chat.completions.create(model=CHAT_MODEL, messages=messages)
    return resp.choices[0].message.content

## 4. Gradio UI

In [ ]:
def chat_ui(message, history):
    history = history or []
    reply = rag_chat(message, history)
    return history + [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]

with gr.Blocks(title="Runbook RAG Chat", theme=gr.themes.Soft()) as demo:
    gr.Markdown("Ask about **deploy**, **restart**, **backup**, or other runbook procedures. Answers are grounded in the knowledge base.")
    chatbot = gr.Chatbot(type="messages", label="Chat")
    msg = gr.Textbox(placeholder="e.g. How do I restart the API?", label="Question")
    submit = gr.Button("Ask")

    submit.click(chat_ui, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)
    msg.submit(chat_ui, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)

demo.launch()